# 05.2 · kv-transfer FAILURE analysis — single examples

Reconstructs instances that FAILED causally in a completed `kv_transfer_eval.py` run (kv-cluster or multi-predicate) and re-runs the same k+v transplant + oracle eval on them, one at a time, for qualitative debugging. Prompt rendering mirrors `05_kv_transfer.ipynb` — the **cached-from** (SRC) and **patched-to** (DST) token windows are marked, including the alignment **pads**. The intervention knobs default to the run_tag's values (`parse_run_tag`) but can be **overridden** below. No hparam search — just eyeballing *why* a case fails.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, glob, re
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4, 7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")
import torch, numpy as np, pandas as pd
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope (incl. align_kv_spans, headwise_kv_patch, ...)
import src.diff.data as D
import src.diff.eval as E
from contextlib import nullcontext

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")
mt = P.mt
GOAL_HEADS, UNION_HEADS = P.GOAL_HEADS, P.UNION_HEADS
READOUT_HEADS = GOAL_HEADS               # goalscope readout heads — always GOAL, kept separate from PATCH
from rich.panel import Panel; from rich.text import Text; from rich.markup import escape

2026-09-06 20:48:17 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Prompt renderer — cached-from / patched-to spans (+ pads)

`show_spans` (same as `05_kv_transfer.ipynb`): the full task text is `bright_yellow`; the exact **KV window** whose k+v are cached/overwritten is `black on dark_orange` (for `align='pad'` the whole span, pads included, is the window); the pivot is `salmon`.

In [2]:
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """task_span -> bright_yellow; kv_span -> black on dark_orange (cached-from / patched-to, incl. pads);
    pivot -> salmon. Priority pivot > kv > task so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos: s = PO + s + PC
        elif i in kv:   s = KO + s + KC
        elif i in task: s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Load failure cases, reconstruct instances, and filter

`rebuild()` reconstructs the kv / multi instance from a record. `filter_fails` narrows the pool by naming / datatype / lang / config / concept-pair; `show_fails` prints the breakdowns.

> **Effective naming:** multi configs were saved with raw naming `l/e/t/r` (the script's `--namings` default is the *string* `"letters"`, iterated into characters by `nargs="+"`). Only the literal `"letters"` renders positional `a/b/c` vars — every other value falls back to **natural** (descriptive) naming. So the completed multi runs are effectively ALL `natural`, which is why the failures cluster there. Filter on the **effective** scheme via `naming="natural"`.

In [3]:
MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"
MECHANISM = "kv_transfer"
_KVROOT = Path(P.env_utils.DEFAULT_RESULTS_DIR) / "diff" / MODEL_SHORT / MECHANISM
ROOTS = {"dual_comp": _KVROOT / "dual_comp", "multi_pred": _KVROOT / "multi_pred"}

def parse_run_tag(t):
    d = {"patch_heads": "goal", "ablate": "off", "ablation_strat": "none", "alpha": 1.0, "align": "pad"}
    for seg in t.split("__"):
        if seg.startswith("ph-"): d["patch_heads"] = seg[3:]
        elif seg.startswith("abl-"):
            rest = seg[4:]
            if rest in ("off", "none"): d["ablate"], d["ablation_strat"] = "off", "none"
            else:
                parts = rest.split("-")
                if parts[-1] in ("zero", "restrict"): d["ablation_strat"] = parts[-1]; parts = parts[:-1]
                d["ablate"] = "-".join(parts)
        elif re.fullmatch(r"a[0-9.]+", seg): d["alpha"] = float(seg[1:])
        elif seg in ("pad", "right"): d["align"] = seg
    return d

def _cluster_by_id(dt, cid):
    return next(c for c in D.KVCLUSTERS[dt] if c.id == cid)

def rebuild(r):
    dt = r["datatype"]
    if r.get("setup") == "dual_comp":
        cl = _cluster_by_id(dt, r["group"])
        return D.KVTransferInstance(cluster=cl, var_name=cl.var_names[0], presentation="description",
            src_lang=r["src_lang"], dst_lang=r["dst_lang"], src_task=r["expected_next"], dst_task=r["dst_next"],
            pivot="auto", meta={"datatype": dt, "cluster": cl.id, "dst_next": r["dst_next"]})
    bank = D.QTASKS[dt]; by = {p.id: p for p in bank.predicates}
    shared_ids = r["group"].split("shared:")[-1].split("-")
    ls = r.get("letter_stub", "indent")
    return D.MultiPredicateInstance(datatype=dt, shared=tuple(by[i] for i in shared_ids),
        src_frontier=by[r["expected_next"]], dst_frontier=by[r["dst_next"]], var_name=bank.var_names[0],
        presentation="listed", src_lang=r["src_lang"], dst_lang=r["dst_lang"],
        naming=r["naming"] if r["naming"] in ("natural", "letters") else "natural",
        letter_stub=ls if ls in ("indent", "var_eq") else "indent",
        meta={"datatype": dt, "dst_next": r["dst_next"], "n_conditions": len(shared_ids) + 1,
              "letter_stub": ls})

def ablate_heads_for(knobs, patch_heads):
    if knobs["ablate"] in ("off", None):
        return []
    lo, hi = (int(x) for x in knobs["ablate"].split("-"))
    allh = [(l, h) for l in range(lo, hi + 1) for h in range(mt.n_heads)]
    return [h for h in allh if h not in set(patch_heads)]

def load_fails(setup, run_tag, causal_only=True):
    """Per-instance records for a run. Default keeps only causal FAILURES (triggered=False).
    Pass causal_only=False to load ALL records that have an oracle verdict -- needed to inspect
    causal SUCCESSES, e.g. the 'triggered but the GoalScope readout did not flip' cases
    (filter_fails(..., causal=True, readout_flipped=False))."""
    out = []
    for rj in (ROOTS[setup] / run_tag).glob("*/results.json"):
        for r in json.loads(rj.read_text()):
            if not r.get("causal"):
                continue
            if causal_only and r["causal"]["triggered"]:
                continue
            r["_cfg"] = rj.parent.name; out.append(r)
    return out

def eff_naming(r):
    """EFFECTIVE variable-naming scheme actually rendered. NOTE: the batch script's --namings default
    is the *string* "letters", which (nargs="+") argparse iterates into characters -> multi configs got
    naming 'l','e','t','r','s'. Only the literal "letters" renders positional a/b/c vars; every other
    value (incl. those stray letters and 'natural') falls back to descriptive/natural naming in
    `var_of`. So filter on this, not the raw stored `naming`."""
    return "letters" if r.get("naming") == "letters" else "natural"

def filter_fails(fails, naming=None, datatype=None, lang=None, config=None, target=None,
                 frontier=None, ro_patched=None, causal=None, readout_flipped=None,
                 letter_stub=None):
    """Narrow the pool. `naming` matches the EFFECTIVE scheme ('natural'|'letters', see eff_naming).
    `ro_patched` filters by the raw patched-readout label ('src'|'dst'|'both'|'neither').
    `letter_stub` filters by the multi-predicate pivot shape ('indent'|'var_eq').

    Two knobs for the intervention/readout cross-tab (need a pool from load_fails(..., causal_only=False)):
      `causal`         -- True = SRC.target triggered, False = not triggered.
      `readout_flipped`-- True = GoalScope's patched readout flipped to the SRC goal (patched=='src');
                          False = it did NOT flip to SRC (patched in {'dst','both','neither'}).
    The 'intervention worked but the readout stayed put' cases are
    filter_fails(pool, causal=True, readout_flipped=False)."""
    def ok(r):
        ro = (r.get("readout") or {}).get("patched")
        trig = (r.get("causal") or {}).get("triggered")
        return (
            (naming is None or eff_naming(r) == naming)
            and (datatype is None or r["datatype"] == datatype)
            and (lang is None or f'{r["src_lang"]}->{r["dst_lang"]}' == lang)
            and (config is None or config in r["_cfg"])
            and (target is None or r["expected_next"] == target)
            and (frontier is None or r["dst_next"] == frontier)
            and (letter_stub is None or r.get("letter_stub") == letter_stub)
            and (ro_patched is None or ro == ro_patched)
            and (causal is None or trig == causal)
            and (readout_flipped is None or (ro is not None and (ro == "src") == readout_flipped)))
    return [r for r in fails if ok(r)]

def show_fails(fails, k=14):
    df = pd.DataFrame([dict(naming=r.get("naming"), eff=eff_naming(r), datatype=r["datatype"],
                            lang=f'{r["src_lang"]}->{r["dst_lang"]}',
                            trig=(r.get("causal") or {}).get("triggered"),
                            ro_patched=(r.get("readout") or {}).get("patched")) for r in fails])
    print(f"{len(fails)} records")
    if len(df):
        if "trig" in df:
            print("  causal triggered :", df.trig.value_counts().to_dict())
        print("  by EFFECTIVE naming:", df.eff.value_counts().to_dict(),
              "  (raw stored naming:", df.naming.value_counts().to_dict(), ")")
        print("  by datatype       :", df.datatype.value_counts().to_dict())
        print("  by patched-readout:", df.ro_patched.value_counts().to_dict())
    for r in fails[:k]:
        ro = r.get("readout") or {}
        print(f"  [{r['_cfg']}] SRC {r['expected_next']!r} -> DST {r['dst_next']!r} "
              f"(naming={eff_naming(r)}) | causal={r['causal']['label']} · "
              f"readout {ro.get('source')}>{ro.get('dest')}>{ro.get('patched')}")
    return fails

SETUP, RUN_TAG = "multi_pred", "ph-goal__abl-33-43-zero__a2__pad"      # <- run to debug
# ALL records (causal HITs + misses). Gate explicitly in filter_fails:
#   causal=False -> the original failure view ; causal=True -> successes.
all_fails = load_fails(SETUP, RUN_TAG, causal_only=False)
show_fails(all_fails, k=4)

362 records
  causal triggered : {True: 232, False: 130}
  by EFFECTIVE naming: {'letters': 362}   (raw stored naming: {'letters': 362} )
  by datatype       : {'list[int]': 128, 'list[str]': 128, 'int': 106}
  by patched-readout: {'src': 177, 'dst': 109, 'neither': 64, 'both': 12}
  [list_int__letters__var_eq__python-cpp] SRC 'all_positive' -> DST 'product_all' (naming=letters) | causal=src · readout src>dst>dst
  [list_int__letters__var_eq__python-cpp] SRC 'mean_int' -> DST 'distinct_count' (naming=letters) | causal=dst · readout src>dst>dst
  [list_int__letters__var_eq__python-cpp] SRC 'product_all' -> DST 'min_val' (naming=letters) | causal=src · readout src>dst>src
  [list_int__letters__var_eq__python-cpp] SRC 'count_even' -> DST 'all_positive' (naming=letters) | causal=neither · readout src>dst>neither


[{'key': 'multi|list[int]|shared=count_odd|src=all_positive|dst=product_all',
  'setup': 'multi_pred',
  'datatype': 'list[int]',
  'group': 'shared:count_odd',
  'naming': 'letters',
  'presentation': 'listed',
  'src_lang': 'python',
  'dst_lang': 'cpp',
  'var_name': 'nums',
  'expected_next': 'all_positive',
  'expected_next_desc': 'whether all values are positive',
  'dst_next': 'product_all',
  'dst_next_desc': 'the product of all values',
  'letter_stub': 'var_eq',
  'raw': {'src_prompt': '<|im_start|>system\nYou are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>\n<|im_start|>user\nWrite a Python program that implements a function `func` which, given a list of integers `nums`, computes:\n(1) the number of odd values\n(2) whether all values are positive\n<|im_end|>\n<|im_start|>assistant\n<code>\ndef func(nums: list[int]):\n    a = sum(1 for x in nums if x % 2 != 0)\n    b =

In [4]:
# Intervention worked (SRC.target triggered) but GoalScope readout did NOT flip to SRC.
# all_fails is now the FULL pool (causal_only=False above), so causal=True works here.
trig_noflip = filter_fails(all_fails, causal=True, readout_flipped=False)   # acted, but probe stayed put
trig_flip   = filter_fails(all_fails, causal=True, readout_flipped=True)    # acted AND probe flipped
print(f"{len(all_fails)} records | causal-HIT & readout stayed: {len(trig_noflip)} | "
      f"causal-HIT & readout flipped: {len(trig_flip)}")
show_fails(trig_noflip, k=8)
# pool = trig_noflip   # <- uncomment to drive the render/re-run cells below off this set

362 records | causal-HIT & readout stayed: 104 | causal-HIT & readout flipped: 128
104 records
  causal triggered : {True: 104}
  by EFFECTIVE naming: {'letters': 104}   (raw stored naming: {'letters': 104} )
  by datatype       : {'list[int]': 44, 'list[str]': 41, 'int': 19}
  by patched-readout: {'dst': 61, 'neither': 37, 'both': 6}
  [list_int__letters__var_eq__python-cpp] SRC 'all_positive' -> DST 'product_all' (naming=letters) | causal=src · readout src>dst>dst
  [list_int__letters__var_eq__python-cpp] SRC 'has_even' -> DST 'distinct_count' (naming=letters) | causal=src · readout src>dst>dst
  [list_int__letters__var_eq__python-cpp] SRC 'has_even' -> DST 'min_val' (naming=letters) | causal=src · readout src>dst>neither
  [list_int__letters__var_eq__python-cpp] SRC 'product_all' -> DST 'all_positive' (naming=letters) | causal=src · readout src>dst>dst
  [list_int__letters__var_eq__python-cpp] SRC 'count_even' -> DST 'mean_int' (naming=letters) | causal=src · readout src>dst>dst
  [

[{'key': 'multi|list[int]|shared=count_odd|src=all_positive|dst=product_all',
  'setup': 'multi_pred',
  'datatype': 'list[int]',
  'group': 'shared:count_odd',
  'naming': 'letters',
  'presentation': 'listed',
  'src_lang': 'python',
  'dst_lang': 'cpp',
  'var_name': 'nums',
  'expected_next': 'all_positive',
  'expected_next_desc': 'whether all values are positive',
  'dst_next': 'product_all',
  'dst_next_desc': 'the product of all values',
  'letter_stub': 'var_eq',
  'raw': {'src_prompt': '<|im_start|>system\nYou are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>\n<|im_start|>user\nWrite a Python program that implements a function `func` which, given a list of integers `nums`, computes:\n(1) the number of odd values\n(2) whether all values are positive\n<|im_end|>\n<|im_start|>assistant\n<code>\ndef func(nums: list[int]):\n    a = sum(1 for x in nums if x % 2 != 0)\n    b =

## Intervention knobs — from the run_tag, override as needed

`KNOBS = parse_run_tag(RUN_TAG)` gives the batch-run settings so the re-run reproduces the recorded verdict. Uncomment any line to override a single knob and probe a variation.

In [28]:
KNOBS = parse_run_tag(RUN_TAG)
# ── optional overrides (uncomment to deviate from the batch run) ──
KNOBS["alpha"] = 2.0                 # OV-boost of PATCH_HEADS on the patched gen/readout
# KNOBS["ablation_strat"] = "restrict" # "zero" | "restrict" | "none"
KNOBS["ablate"] = "33-43"            # ablate layer range, or "off"
# KNOBS["align"] = "pad"               # "pad" | "right"
KNOBS["patch_heads"] = "goal"       # "goal" | "union"

# PATCH_HEADS = GOAL_HEADS if KNOBS["patch_heads"] == "goal" else UNION_HEADS
with open(
    os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR), "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")
) as f:
    PATCH_HEADS = [(layer_idx, head_idx) for layer_idx, head_idx in json.load(f)]
    READOUT_HEADS = PATCH_HEADS

ABLATE_HEADS = ablate_heads_for(KNOBS, PATCH_HEADS)
print(f"SETUP={SETUP} · RUN_TAG={RUN_TAG}")
print(f"KNOBS={KNOBS}")
print(f"patch_heads={len(PATCH_HEADS)} · readout_heads={len(READOUT_HEADS)} · ablate_heads={len(ABLATE_HEADS)}")

SETUP=multi_pred · RUN_TAG=ph-goal__abl-33-43-zero__a2__pad
KNOBS={'patch_heads': 'goal', 'ablate': '33-43', 'ablation_strat': 'zero', 'alpha': 2.0, 'align': 'pad'}
patch_heads=250 · readout_heads=250 · ablate_heads=379


## Select a failure and render the cached-from / patched-to spans

Filter the pool (e.g. `naming='natural'`), then pick by index or concept pair. The render uses `align_kv_spans(mode=KNOBS['align'])` — the same alignment the runner applies — so the orange window shows exactly which tokens (incl. pads) are cached from SRC and overwritten in DST.

In [43]:
# filter the pool, then pick one
pool = filter_fails(
    all_fails, 
    naming="letters", 
    causal = False, 
    # readout_flipped=False,
    letter_stub="indent"
)
print(f"{len(pool)} failures after filter (of {len(all_fails)})")

def pick(fs, i=0): return fs[i % len(fs)]
def pick_by_pair(fs, target, frontier):
    hits = [r for r in fs if r["expected_next"] == target and r["dst_next"] == frontier]
    return hits[0] if hits else None

rec = pick(pool, i=8)                                  # <- change i, or pick_by_pair(pool, "...", "...")
inst = rebuild(rec)
m = D.materialize(inst)
SRC_SPAN0 = token_span(m["src_prompt"], m["src_task_text"])   # pre-align located spans
DST_SPAN0 = token_span(m["dst_prompt"], m["dst_task_text"])
# align -> padded tokens + cached-from (SRC_POS) / patched-to (DST_POS) windows
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], SRC_SPAN0, DST_SPAN0, mode=KNOBS["align"],
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
piv_src, piv_dst = src_tok["input_ids"].shape[1] - 1, dst_tok["input_ids"].shape[1] - 1

print(f"[{rec['_cfg']}] naming={rec.get('naming')}  SRC.target={inst.expected_next!r}  <-  DST.frontier={inst.dst_next!r}")
print(f"  {inst.expected_next_desc!r}  vs  {inst.dst_next_desc!r}")
print(f"OLD batch verdict: causal={rec['causal']['label']} (FAILED) · readout_patched={rec.get('readout', {}).get('patched')}")
show_spans(src_tok, f"SRC · CACHED-FROM {tuple(SRC_SPAN)} (orange) · pivot@{piv_src} · align={KNOBS['align']}",
           task_span=SRC_SPAN, kv_span=(SRC_POS[0], SRC_POS[-1] + 1), piv=-1)
show_spans(dst_tok, f"DST · PATCHED-TO {tuple(DST_SPAN)} (orange) · pivot@{piv_dst} · align={KNOBS['align']}",
           task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), piv=-1)

87 failures after filter (of 362)
[list_str__letters__indent__python-python] naming=letters  SRC.target='distinct_count'  <-  DST.frontier='count_long'
  'the number of distinct words'  vs  'the number of words longer than five characters'
OLD batch verdict: causal=neither (FAILED) · readout_patched=src


╭─ SRC · CACHED-FROM (63, 84) (orange) · pivot@112 · align=pad ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the length of the shortest word\n                                                                           │
│ (2) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = min(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · PATCHED-TO (63, 84) (orange) · pivot@112 · align=pad ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the length of the shortest word\n                                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = min(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Re-run the intervention — SRC / DST / patched generations + readouts

`run_kv_transfer_intervention` with `KNOBS` (it re-derives the same alignment internally). Grouped per run (red=source, green=destination, salmon=patched). Eyeball whether the patched DST's first computation jumps to the SRC target, reverts to its own frontier, or goes hybrid.

In [ ]:
raw = P.run_kv_transfer_intervention(
    mt, inst, PATCH_HEADS, readout_heads=READOUT_HEADS,
    alpha=KNOBS["alpha"], align=KNOBS["align"],
    # ablate_heads=ABLATE_HEADS, 
    # ablation_strat=KNOBS["ablation_strat"], 
    max_new=128
)

rich_print(f"[red1]source_run · clean SRC (target {inst.expected_next!r}):[/red1] \n{escape(raw['gen_src'])}")
rich_print(f"   ↳ goalscope · source_run: [italic]{escape(raw['ro_src'])}[/italic]")
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.dst_next!r}):[/sea_green3] \n{escape(raw['gen_dst'])}")
rich_print(f"   ↳ goalscope · destination_run: [italic]{escape(raw['ro_dst'])}[/italic]")
rich_print(f"[salmon1]patched_run · DST (expect SRC target {inst.expected_next!r}):[/salmon1] \n{escape(raw['gen_patched'])}")
rich_print(f"   ↳ goalscope · patched_run: [italic]{escape(raw['ro_patched'])}[/italic]")

ev = E.eval_kv_transfer(inst, patched_gen=raw["gen_patched"], readouts=(raw["ro_src"], raw["ro_dst"], raw["ro_patched"]))
rich_print(f"[bold]EVAL[/bold] · {ev.summary()}")
rich_print("  causal :", ev.causal)
rich_print("  readout:", ev.readout)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=112 '    ' => Identifying unique elements and counting them.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=112 '    ' => I'm counting lines where the length of the string is greater than 5.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=112 '    ' => Identifying unique elements in a list.

... (6 outputs left out)

EVAL · causal: TRIGGERED (src) | readout: src→dst→src

causal :
{
    'label': 'src',
    'why': 'The first computation `len(set(words))` counts distinct words (unique elements via set), which is 
sub-goal A.',
    'raw': '<verdict>A</verdict>\n<why>The first computation `len(set(words))` counts distinct words (unique 
elements via set), which is sub-goal A.</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase "identifying unique elements and counting them" directly describes computing 
distinct/unique words and their count, which corresponds to sub-goal A (number of distinct words).',
            'raw': '<verdict>A</verdict>\n<why>The phrase "identifying unique elements and counting them" directly 
describes computing distinct/unique words and their count, which corresponds to sub-goal A (number of distinct 
words).</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly describes counting items where string length is greater than 5, which is 
the computation for sub-goal B (words longer than five characters).',
            'raw': '<verdict>B</verdict>\n<why>The phrase explicitly describes counting items where string length 
is greater than 5, which is the computation for sub-goal B (words longer than five characters).</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'Identifying unique elements corresponds to finding distinct words, which is sub-goal A.',
            'raw': '<verdict>A</verdict><why>Identifying unique elements corresponds to finding distinct words, 
which is sub-goal A.</why>'
        }
    }
}

: 

## Batch of (filtered) failures — quick re-run table

Re-run the first N failures from the current `pool` (respects the filter + KNOBS) and tabulate old vs new verdict + a patched-gen snippet, to spot patterns (revert-to-DST, hybrid, incoherent).

In [37]:
# N = 8
# rows = []
# for r in pool[:N]:
#     it = rebuild(r)
#     rw = P.run_kv_transfer_intervention(
#         mt, it, PATCH_HEADS, readout_heads=READOUT_HEADS, alpha=KNOBS["alpha"], align=KNOBS["align"],
#         ablate_heads=ABLATE_HEADS, ablation_strat=KNOBS["ablation_strat"], max_new=64)
#     e = E.eval_kv_transfer(it, patched_gen=rw["gen_patched"], readouts=(rw["ro_src"], rw["ro_dst"], rw["ro_patched"]))
#     rows.append(dict(
#         cfg=r["_cfg"].replace(f"{SETUP}__", "").replace("__python-python", "").replace("__python-cpp", "->cpp"),
#         naming=r.get("naming"), SRC=it.expected_next, DST=it.dst_next, old=r["causal"]["label"],
#         new=e.causal["label"], new_trig=e.causal["triggered"], ro_pat=e.readout["patched"],
#         gen=rw["gen_patched"][:50].replace(chr(10), " ")))
# res = pd.DataFrame(rows)
# print(f"re-ran {len(res)} failures ({SETUP} · {RUN_TAG} · KNOBS alpha={KNOBS['alpha']} strat={KNOBS['ablation_strat']}) "
#       f"· new triggers: {int(res.new_trig.sum())}")
# res